# Giai đoạn 1 – Thu thập dữ liệu: Job postings (123job.vn)
**Dự án:** A Day in the Life of a Data Analyst – VTI Academy

**Mục tiêu:** cào tin tuyển dụng IT từ **123job.vn**, tách kỹ năng (ngôn ngữ, database, framework, nền tảng/công cụ), rồi lưu CSV để ghép với khảo sát Stack Overflow.

**Thứ tự chạy:** chạy lần lượt từng ô từ trên xuống.
1. Ô cài đặt / import / cấu hình / hàm
2. `check` – xác nhận trang cào được
3. `discover` – xem dạng link tin chi tiết
4. `scrape` – cào thật → `jobs_raw.csv`, `job_skills.csv`
5. Xem kết quả và lưu local

> **Đã sửa:** cài thư viện an toàn trên Linux PEP 668; lấy **tên công ty** từ JSON-LD; lọc tin không phải IT; gom nhiều trang danh sách vì `?page=` của 123job thường trả về cùng trang 1 khi gọi bằng Requests.


## 1. Cài đặt thư viện

In [1]:
# Cài thư viện thiếu — không dừng notebook nếu pip lỗi (PEP 668 / Linux).
import importlib
import subprocess
import sys

_pkgs = [
    ("lxml", "lxml"),
    ("bs4", "beautifulsoup4"),
    ("requests", "requests"),
    ("pandas", "pandas"),
]

def _missing():
    miss = []
    for mod, pip_name in _pkgs:
        try:
            importlib.import_module(mod)
        except ImportError:
            miss.append(pip_name)
    return miss

missing = _missing()
if not missing:
    print("Đủ thư viện:", ", ".join(m for m, _ in _pkgs))
else:
    print("Thiếu:", ", ".join(missing), "— thử pip...")
    for args in (
        [sys.executable, "-m", "pip", "install", "-q", *missing],
        [sys.executable, "-m", "pip", "install", "-q", "--user", *missing],
        [sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *missing],
    ):
        r = subprocess.run(args, capture_output=True, text=True)
        if r.returncode == 0 and not _missing():
            print("Đã cài bằng:", " ".join(args[3:6]), "...")
            break
    else:
        still = _missing()
        if still:
            print("CẢNH BÁO: vẫn thiếu", still, "- nếu import lỗi, cài tay trong venv.")
        else:
            print("Đã có đủ thư viện sau khi thử pip.")


Đủ thư viện: lxml, bs4, requests, pandas


## 2. Import thư viện

In [2]:
import re
import sys
import json
import time
import random
from urllib import robotparser
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup


## 3. Cấu hình
**123job.vn** (đã chọn sau `check`). Ghi chú:
- `LIST_URLS`: nhiều trang danh sách IT (vì phân trang `page=2,3,...` thường không đổi kết quả khi cào bằng Requests)
- `JOB_LINK_REGEX`: link tin chi tiết dạng `/viec-lam/<slug>`
- `MAX_JOBS`: giới hạn khi chạy thử (`None` = lấy hết link đã gom)


In [3]:
# ============================================================
# 1. CẤU HÌNH  — 123job.vn
# ============================================================
BASE_URL = "https://123job.vn"

# FIX: page=2+ thường trùng page=1 khi Requests; gom nhiều nguồn IT thay vì chỉ tăng page
LIST_URLS = [
    # IT phần mềm — nhiều trang (page= đôi khi vẫn trả tin mới)
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=1",
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=2",
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=3",
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=4",
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=5",
    # IT phần cứng / mạng
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-cứng-mạng",
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-cứng-mạng?page=2",
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-cứng-mạng?page=3",
    # Ngành IT-adjacent
    "https://123job.vn/nganh-nghe/việc-làm-games",
    "https://123job.vn/nganh-nghe/việc-làm-công-nghệ-cao",
    "https://123job.vn/nganh-nghe/việc-làm-điện-tử-viễn-thông",
    "https://123job.vn/nganh-nghe/việc-làm-thiết-kế-đồ-họa-web",
    # Tìm theo từ khóa IT
    "https://123job.vn/tuyen-dung?q=developer",
    "https://123job.vn/tuyen-dung?q=developer&page=2",
    "https://123job.vn/tuyen-dung?q=l%E1%BA%ADp+tr%C3%ACnh",
    "https://123job.vn/tuyen-dung?q=l%E1%BA%ADp+tr%C3%ACnh&page=2",
    "https://123job.vn/tuyen-dung?q=python",
    "https://123job.vn/tuyen-dung?q=java",
    "https://123job.vn/tuyen-dung?q=react",
    "https://123job.vn/tuyen-dung?q=php",
    "https://123job.vn/tuyen-dung?q=devops",
    "https://123job.vn/tuyen-dung?q=data+analyst",
    "https://123job.vn/tuyen-dung?q=backend",
    "https://123job.vn/tuyen-dung?q=frontend",
    "https://123job.vn/tuyen-dung?q=mobile+developer",
    "https://123job.vn/tuyen-dung?q=software+engineer",
    "https://123job.vn/tuyen-dung?q=tester",
    "https://123job.vn/tuyen-dung?q=devops+engineer",
]

# Giữ LIST_URL + {page} để discover/scrape tương thích (và ô kiểm tra phân trang)
LIST_URL = (
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm"
    "?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page={page}"
)

JOB_LINK_REGEX = r"^/viec-lam/[^/]+$"   # /viec-lam/<slug>
MAX_PAGES = 2                           # chỉ dùng nếu LIST_URLS rỗng
MAX_JOBS = None                           # None = cào hết link đã gom (không giới hạn)
DELAY = (1.5, 2.5)

# Selector dự phòng (ưu tiên JSON-LD JobPosting trong scrape)
SELECTORS = {
    "title": "h1",
    "company": ".company-name h2",      # FIX: tên công ty trên 123job
    "location": None,                   # lấy từ JSON-LD
    "description": None,
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; VTI-Academy-StudentProject/1.0)",
    "Accept-Language": "vi,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


## 4. Từ điển kỹ năng và các hàm xử lý

In [4]:
# ============================================================
# 2. TỪ ĐIỂN KỸ NĂNG  (chuẩn hóa tên: nhiều cách viết -> 1 tên chuẩn)
# ============================================================
SKILLS = {
    "language": {
        "JavaScript": ["javascript", "node.js", "nodejs"],
        "TypeScript": ["typescript"],
        "Python": ["python"],
        "Java": ["java"],
        "C#": ["c#", "csharp"],
        "C++": ["c++"],
        "PHP": ["php"],
        "Go": ["golang", "go lang"],
        "Kotlin": ["kotlin"],
        "Swift": ["swift"],
        "Ruby": ["ruby"],
        "Rust": ["rust"],
        "SQL": ["sql", "t-sql", "pl/sql"],
        "HTML/CSS": ["html", "css", "html5", "css3"],
        "Bash/Shell": ["bash", "shell script"],
        "R": ["rstudio"],
    },
    "database": {
        "MySQL": ["mysql"],
        "PostgreSQL": ["postgresql", "postgres"],
        "SQL Server": ["sql server", "mssql", "ms sql"],
        "Oracle": ["oracle"],
        "MongoDB": ["mongodb", "mongo"],
        "Redis": ["redis"],
        "SQLite": ["sqlite"],
        "Elasticsearch": ["elasticsearch"],
        "Cassandra": ["cassandra"],
        "MariaDB": ["mariadb"],
    },
    "framework": {
        "React": ["react", "reactjs", "react.js"],
        "Angular": ["angular", "angularjs"],
        "Vue.js": ["vue", "vuejs", "vue.js"],
        "Spring": ["spring boot", "spring"],
        "Django": ["django"],
        "Flask": ["flask"],
        "ASP.NET": ["asp.net"],
        ".NET": [".net", "dotnet", ".net core"],
        "Laravel": ["laravel"],
        "Express": ["expressjs", "express.js"],
        "Pandas": ["pandas"],
        "TensorFlow": ["tensorflow"],
        "PyTorch": ["pytorch"],
    },
    "platform_tool": {
        "AWS": ["aws", "amazon web services"],
        "Azure": ["azure"],
        "Google Cloud": ["gcp", "google cloud"],
        "Docker": ["docker"],
        "Kubernetes": ["kubernetes", "k8s"],
        "Linux": ["linux"],
        "Git": ["git", "github", "gitlab"],
        "Jira": ["jira"],
        "Power BI": ["power bi", "powerbi"],
        "Tableau": ["tableau"],
        "Android": ["android"],
        "iOS": ["ios"],
    },
}


def build_patterns():
    """Lookaround: 'java' không khớp trong 'javascript'; 'sql' không khớp trong 'mysql'."""
    patterns = []
    for group, mapping in SKILLS.items():
        for canonical, aliases in mapping.items():
            for alias in aliases:
                rx = re.compile(
                    r"(?<![A-Za-z0-9+#.])" + re.escape(alias) + r"(?![A-Za-z0-9+#])",
                    flags=re.IGNORECASE,
                )
                patterns.append((group, canonical, rx))
    return patterns


PATTERNS = build_patterns()


def extract_skills(text):
    """Trả về dict {nhóm: [tên kỹ năng chuẩn,...]} (không trùng)."""
    found = {g: set() for g in SKILLS}
    for group, canonical, rx in PATTERNS:
        if rx.search(text or ""):
            found[group].add(canonical)
    return {g: sorted(v) for g, v in found.items()}


# FIX: lọc tin không phải IT (vd Account Intern / C&B bị gắn nhầm ngành IT trên 123job)
IT_TITLE_KW = [
    "developer", "engineer", "devops", "programmer", "lập trình", "lap trinh",
    "software", "backend", "frontend", "fullstack", "full-stack", "full stack",
    "mobile", "android", "ios", "data analyst", "data scientist", "data engineer",
    "machine learning", "ai engineer", "llm", "qa", "tester", "automation",
    "cloud", "security", "sysadmin", "system", "network", "database", "helpdesk",
    "help desk", "it support", "nhân viên it", "nv it", "phụ trách it", "phu trach it", "kỹ thuật viên",
    "ki thuat vien", "business analyst", "solution architect", "kiến trúc",
    "phần mềm", "phần cứng", "iot", "embedded", "firmware", "blockchain", "golang",
    "python", "java", "react", "node", "php", "vue", "angular", ".net",
    " sap", "erp", "cto", "sre", "platform", "infra", "plc", "robot",
]
NON_IT_TITLE_KW = [
    "account intern", "chuyên viên c&b", "c&b ", " nhân sự", "kế toán",
    "sales executive", "nhân viên kinh doanh", "tạp vụ", "lễ tân",
    "receptionist", "content marketing", "brand marketing",
]


def is_it_job(title, skills):
    """Giữ tin nếu có kỹ năng tech hoặc tiêu đề mang tính IT; loại rõ ràng non-IT khi không có skill."""
    t = (title or "").lower()
    n_skills = sum(len(skills.get(g, [])) for g in ("language", "database", "framework", "platform_tool"))
    if n_skills >= 1:
        return True
    if any(k in t for k in NON_IT_TITLE_KW):
        return False
    return any(k in t for k in IT_TITLE_KW)


In [5]:
# ============================================================
# 3. HÀM TIỆN ÍCH CHO VIỆC CÀO
# ============================================================
session = requests.Session()
session.headers.update(HEADERS)

_robots_cache = {"base": None, "rp": None}


def _load_robots():
    if _robots_cache["base"] == BASE_URL and _robots_cache["rp"] is not None:
        return
    rp = robotparser.RobotFileParser()
    rp.set_url(urljoin(BASE_URL, "/robots.txt"))
    try:
        rp.read()
    except Exception:
        rp = None
    _robots_cache.update({"base": BASE_URL, "rp": rp})


def allowed_by_robots(url):
    """Tôn trọng robots.txt (123job Disallow: /ajax, /apply)."""
    try:
        _load_robots()
        rp = _robots_cache["rp"]
        if rp is None:
            return True
        return rp.can_fetch(HEADERS["User-Agent"], url)
    except Exception:
        return True


def get_html(url):
    """Tải 1 trang + delay. Trả về None nếu lỗi/bị chặn/robots."""
    if not allowed_by_robots(url):
        print(f"  [robots] bỏ: {url}")
        return None
    try:
        resp = session.get(url, timeout=20)
    except requests.RequestException as e:
        print(f"  [LỖI mạng] {url}: {e}")
        return None
    time.sleep(random.uniform(*DELAY))
    if resp.status_code != 200:
        print(f"  [HTTP {resp.status_code}] {url}")
        return None
    return resp.text


def select_text(soup, selector):
    if not selector:
        return ""
    el = soup.select_one(selector)
    return el.get_text(" ", strip=True) if el else ""


def parse_job_ld(soup):
    """FIX: đọc JSON-LD JobPosting (company, location, description, industry)."""
    for script in soup.find_all("script", attrs={"type": "application/ld+json"}):
        raw = script.string or script.get_text() or ""
        try:
            data = json.loads(raw)
        except Exception:
            continue
        if isinstance(data, dict) and data.get("@type") == "JobPosting":
            return data
        if isinstance(data, list):
            for item in data:
                if isinstance(item, dict) and item.get("@type") == "JobPosting":
                    return item
    return {}


def company_from_page(soup, ld):
    org = (ld or {}).get("hiringOrganization") or {}
    if isinstance(org, dict) and org.get("name"):
        return str(org["name"]).strip()
    el = soup.select_one(".company-name h2") or soup.select_one(".company-name")
    if el:
        return el.get_text(" ", strip=True).split("Quy mô")[0].strip()
    return select_text(soup, SELECTORS.get("company"))


def location_from_page(soup, ld):
    loc = (ld or {}).get("jobLocation")
    if isinstance(loc, dict):
        addr = loc.get("address") or {}
        if isinstance(addr, dict):
            parts = [addr.get("streetAddress"), addr.get("addressLocality"), addr.get("addressRegion")]
            parts = [p for p in parts if p]
            # streetAddress trên 123job đôi khi đã gồm đủ
            if parts:
                # bỏ trùng
                text = ", ".join(dict.fromkeys(parts))
                return text
    return select_text(soup, SELECTORS.get("location"))


def get_description(soup, ld=None):
    if ld and ld.get("description"):
        # description JSON-LD có HTML
        return BeautifulSoup(ld["description"], "lxml").get_text(" ", strip=True)
    text = select_text(soup, SELECTORS.get("description"))
    if text:
        return text
    for tag in ("main", "article"):
        el = soup.find(tag)
        if el:
            return el.get_text(" ", strip=True)
    return soup.get_text(" ", strip=True)


def iter_list_urls(max_pages):
    urls = globals().get("LIST_URLS") or []
    if urls:
        return list(urls)
    return [LIST_URL.format(page=p) for p in range(1, max_pages + 1)]


def collect_job_links(max_pages):
    """Gom link tin chi tiết từ các trang danh sách (không trùng)."""
    links, seen = [], set()
    link_rx = re.compile(JOB_LINK_REGEX)
    for url in iter_list_urls(max_pages):
        print(f"Trang danh sách: {url}")
        html = get_html(url)
        if html is None:
            continue
        soup = BeautifulSoup(html, "lxml")
        new = 0
        for a in soup.find_all("a", href=True):
            full = urljoin(BASE_URL, a["href"]).split("?")[0]
            if link_rx.search(urlparse(full).path) and full not in seen:
                seen.add(full)
                links.append(full)
                new += 1
        print(f"  +{new} link mới (tổng {len(links)})")
    return links


In [6]:
# ============================================================
# 4. discover + scrape  (check nằm ở ô riêng bên dưới)
# ============================================================
def discover():
    """In link trên trang danh sách đầu để chỉnh JOB_LINK_REGEX."""
    first = (globals().get("LIST_URLS") or [LIST_URL.format(page=1)])[0]
    html = get_html(first)
    if html is None:
        print("Không tải được trang danh sách.")
        return
    soup = BeautifulSoup(html, "lxml")
    hrefs = sorted({urljoin(BASE_URL, a["href"]).split("?")[0] for a in soup.find_all("a", href=True)})
    print(f"Có {len(hrefs)} link. 40 link đầu:")
    for h in hrefs[:40]:
        print(" ", h)
    link_rx = re.compile(JOB_LINK_REGEX)
    matched = [h for h in hrefs if link_rx.search(urlparse(h).path)]
    print(f"\nJOB_LINK_REGEX khớp {len(matched)} link. Ví dụ:")
    for h in matched[:10]:
        print(" ", h)


def _empty_csvs():
    pd.DataFrame(columns=[
        "job_url", "job_title", "company", "location", "industry",
        "language_list", "database_list", "framework_list",
        "platform_tool_list", "description_length",
    ]).to_csv("jobs_raw.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(columns=["job_url", "skill_group", "skill"]).to_csv(
        "job_skills.csv", index=False, encoding="utf-8-sig"
    )


def scrape():
    """Danh sách -> chi tiết -> tách kỹ năng / lọc IT -> CSV."""
    seed = (globals().get("LIST_URLS") or [LIST_URL.format(page=1)])[0]
    if not allowed_by_robots(seed):
        print("robots.txt KHÔNG cho phép cào trang seed. Dừng.")
        _empty_csvs()
        return

    links = collect_job_links(MAX_PAGES)
    max_jobs = globals().get("MAX_JOBS")
    if max_jobs is not None:
        links = links[: int(max_jobs)]
    print(f"\nTổng {len(links)} tin sẽ cào chi tiết (sau giới hạn MAX_JOBS).\n")

    rows, long_rows = [], []
    skipped_non_it = 0
    for i, url in enumerate(links, 1):
        html = get_html(url)
        if html is None:
            continue
        soup = BeautifulSoup(html, "lxml")
        ld = parse_job_ld(soup)

        title = (ld.get("title") if ld else None) or select_text(soup, SELECTORS["title"])
        company = company_from_page(soup, ld)
        location = location_from_page(soup, ld)
        industry = (ld.get("industry") if ld else None) or ""
        desc = get_description(soup, ld)

        skills = extract_skills(f"{title} {desc}")
        if not is_it_job(title, skills):
            skipped_non_it += 1
            if i <= 5 or skipped_non_it <= 3:
                print(f"  [bỏ non-IT] {title}")
            continue

        rows.append({
            "job_url": url,
            "job_title": title,
            "company": company,
            "location": location,
            "industry": industry,
            "language_list": ";".join(skills["language"]),
            "database_list": ";".join(skills["database"]),
            "framework_list": ";".join(skills["framework"]),
            "platform_tool_list": ";".join(skills["platform_tool"]),
            "description_length": len(desc),
        })
        for group, names in skills.items():
            for name in names:
                long_rows.append({"job_url": url, "skill_group": group, "skill": name})

        if i % 10 == 0 or i == len(links):
            print(f"  đã xử lý {i}/{len(links)} URL | giữ {len(rows)} tin IT")
            pd.DataFrame(rows).to_csv("jobs_raw.csv", index=False, encoding="utf-8-sig")

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset="job_url")
    else:
        _empty_csvs()
    df.to_csv("jobs_raw.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(long_rows).to_csv("job_skills.csv", index=False, encoding="utf-8-sig")
    print(f"\nXong: {len(df)} tin IT -> jobs_raw.csv | {len(long_rows)} dòng kỹ năng -> job_skills.csv")
    print(f"Đã bỏ {skipped_non_it} tin không đủ dấu hiệu IT.")


## 5. `check` – chọn trang cào được
Mở trình duyệt, vào 2–3 trang tuyển dụng IT (TopCV, TopDev, VietnamWorks, CareerViet...), tìm việc lập trình, **copy URL trang kết quả** rồi dán vào danh sách bên dưới.

Trang nào hiện **`OK -> có thể thử cào bằng Requests`** thì chọn trang đó. Trang bị chặn hoặc robots.txt không cho phép thì bỏ.

In [7]:
def check(urls):
    """Thử nhiều URL: robots + HTTP + dấu hiệu chặn bot."""
    for url in urls:
        parts = urlparse(url)
        origin = f"{parts.scheme}://{parts.netloc}"

        rp = robotparser.RobotFileParser()
        rp.set_url(origin + "/robots.txt")
        try:
            rp.read()
            robots_ok = rp.can_fetch(HEADERS["User-Agent"], url)
        except Exception:
            robots_ok = None

        try:
            resp = session.get(url, timeout=20)
        except requests.RequestException as e:
            print(f"\n{url}\n  KHÔNG KẾT NỐI ĐƯỢC: {e}")
            continue

        soup = BeautifulSoup(resp.text, "lxml")
        title = soup.title.get_text(strip=True) if soup.title else "(không có title)"
        n_links = len(soup.find_all("a", href=True))
        for t in soup(["script", "style", "noscript"]):
            t.decompose()
        visible = soup.get_text(" ", strip=True).lower()

        block_words = ["just a moment", "verify you are human", "access denied",
                       "attention required", "checking your browser"]
        hit = [w for w in block_words if w in title.lower() or w in visible[:1500]]

        if resp.status_code != 200 or hit:
            verdict = f"BỊ CHẶN ({resp.status_code}, {hit}) -> bỏ trang này"
        elif robots_ok is False:
            verdict = "robots.txt KHÔNG cho phép -> bỏ trang này"
        elif len(visible) < 500 or n_links < 30:
            verdict = "ít nội dung, có thể do JavaScript tạo -> cần Selenium/Playwright"
        else:
            verdict = "OK -> có thể thử cào bằng Requests"
        print(f"\n{url}\n  Title: {title}\n  HTTP {resp.status_code} | robots: {robots_ok} | "
              f"{n_links} link | {len(visible)} ký tự chữ\n  => {verdict}")
        time.sleep(random.uniform(*DELAY))


In [8]:
URLS_TO_CHECK = [
    "https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm",
    "https://123job.vn/tuyen-dung?q=developer",
    "https://www.topcv.vn/tim-viec-lam-developer?type_keyword=1&sba=1",  # thường bị chặn — để so sánh
]
check(URLS_TO_CHECK)



https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm
  Title: 3500+ Việc làm ngành IT phần mềm Mới Nhất 10/2026 | 123job
  HTTP 200 | robots: True | 282 link | 20032 ký tự chữ
  => OK -> có thể thử cào bằng Requests



https://123job.vn/tuyen-dung?q=developer
  Title: 1500+ Việc làm, tuyển dụng Developer - Nhiều phúc lợi 2026 | 123job
  HTTP 200 | robots: True | 268 link | 19334 ký tự chữ
  => OK -> có thể thử cào bằng Requests



https://www.topcv.vn/tim-viec-lam-developer?type_keyword=1&sba=1
  Title: Just a moment...
  HTTP 403 | robots: False | 0 link | 16 ký tự chữ
  => BỊ CHẶN (403, ['just a moment']) -> bỏ trang này


## 6. Cập nhật cấu hình
Cấu hình 123job đã sẵn. Nếu đổi `LIST_URLS` / `MAX_JOBS`, chạy lại ô cấu hình và các ô hàm bên dưới trước khi `scrape`.


## 7. `discover` – dò dạng link tin chi tiết
In ra các link trên trang danh sách đầu tiên. Tìm phần chung của các link **tin tuyển dụng cụ thể** để điền `JOB_LINK_REGEX`. Không chắc thì copy kết quả hỏi lại.

In [9]:
discover()

Có 188 link. 40 link đầu:
  http://123job.vn/nganh-nghe/vi%E1%BB%87c-l%C3%A0m-it-ph%E1%BA%A7n-m%E1%BB%81m
  https://123job.vn
  https://123job.vn/
  https://123job.vn/alerts
  https://123job.vn/auth/forgot-password
  https://123job.vn/auth/login
  https://123job.vn/auth/logout
  https://123job.vn/auth/register
  https://123job.vn/bai-viet
  https://123job.vn/bai-viet/bang-mo-ta-cong-viec-56c
  https://123job.vn/bai-viet/cam-nang-tuyen-dung-58c
  https://123job.vn/bai-viet/con-duong-su-nghiep-10c
  https://123job.vn/bai-viet/dinh-huong-nghe-nghiep-31c
  https://123job.vn/bai-viet/thanh-cong-trong-cong-viec-24c
  https://123job.vn/bai-viet/tim-viec-30c
  https://123job.vn/bai-viet/xay-dung-doanh-nghiep-55c
  https://123job.vn/browse
  https://123job.vn/browse/company
  https://123job.vn/browse/industrial-area
  https://123job.vn/company
  https://123job.vn/company/search
  https://123job.vn/cover-letter/mẫu-cover-letter
  https://123job.vn/cv/guide
  https://123job.vn/cv/keywords
  https

## 8. `scrape` – cào dữ liệu thật
Chạy chậm là bình thường (có nghỉ 2–4 giây mỗi request để lịch sự với server). Đừng tắt giữa chừng.

In [10]:
scrape()

Trang danh sách: https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=1


  +30 link mới (tổng 30)
Trang danh sách: https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=2


  +0 link mới (tổng 30)
Trang danh sách: https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=3


  +0 link mới (tổng 30)
Trang danh sách: https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=4


  +0 link mới (tổng 30)
Trang danh sách: https://123job.vn/nganh-nghe/việc-làm-it-phần-mềm?cat=IT+phần+mềm&sort=new&cat_name=IT+phần+mềm&page=5


  +0 link mới (tổng 30)
Trang danh sách: https://123job.vn/nganh-nghe/việc-làm-it-phần-cứng-mạng


... (65 outputs left out)

  đã xử lý 480/509 URL | giữ 401 tin IT


  đã xử lý 490/509 URL | giữ 411 tin IT


  đã xử lý 500/509 URL | giữ 421 tin IT


  đã xử lý 509/509 URL | giữ 430 tin IT

Xong: 430 tin IT -> jobs_raw.csv | 1212 dòng kỹ năng -> job_skills.csv
Đã bỏ 79 tin không đủ dấu hiệu IT.


In [11]:
# Kiểm tra phân trang 123job (thường page=2 trùng page=1 khi dùng Requests)
def get_links(url):
    h = get_html(url)
    if not h:
        return None, set()
    s = BeautifulSoup(h, "lxml")
    out = set()
    for a in s.find_all("a", href=True):
        full = urljoin(BASE_URL, a["href"]).split("?")[0]
        if re.search(JOB_LINK_REGEX, urlparse(full).path):
            out.add(full)
    return s, out

s1, l1 = get_links(LIST_URL.format(page=1))
s2, l2 = get_links(LIST_URL.format(page=2))
print("Trang 1:", len(l1), "link | Trang 2:", len(l2), "link")
print("Trùng nhau:", len(l1 & l2), "| chỉ có ở trang 2:", len(l2 - l1))
print("Title trang 2:", s2.title.get_text(strip=True) if s2 and s2.title else None)
print("=> Nếu trùng ~100%: dùng LIST_URLS (nhiều ngành/từ khóa) thay vì tăng MAX_PAGES.")


Trang 1: 30 link | Trang 2: 30 link
Trùng nhau: 30 | chỉ có ở trang 2: 0
Title trang 2: 3500+ Việc làm ngành IT phần mềm Mới Nhất 10/2026 | 123job
=> Nếu trùng ~100%: dùng LIST_URLS (nhiều ngành/từ khóa) thay vì tăng MAX_PAGES.


## 9. Xem thử kết quả và lưu

In [12]:
from pathlib import Path
if not Path("jobs_raw.csv").exists():
    print("Chưa có jobs_raw.csv — chạy scrape() trước.")
else:
    df = pd.read_csv("jobs_raw.csv")
    print("Số tin cào được:", len(df))
    print("Company non-null:", int(df["company"].notna().sum()) if "company" in df.columns else "n/a")
    print(df.head(10).to_string())


Số tin cào được: 430
Company non-null: 425
                                                                           job_url                                        job_title                                                            company                                         location     industry        language_list                database_list framework_list platform_tool_list  description_length
0        https://123job.vn/viec-lam/ki-thuat-vien-phan-cung-it-helpdesk-g9d1zx28rk             Kĩ Thuật Viên Phần Cứng/ IT Helpdesk                                      công ty cổ phần công nghệ tmt                                        Hải Phòng  IT phần mềm                  NaN                          NaN            NaN                NaN                2837
1  https://123job.vn/viec-lam/ky-su-giai-phap-web-full-stack-figma-saas-lroWzKRar4  Kỹ Sư Giải Pháp Web – Full-Stack / Figma / SaaS             Công ty Cổ Phần Giải Pháp Phần mềm Trực tuyến Việt Nam                             

In [13]:
from pathlib import Path
if not Path("job_skills.csv").exists():
    print("Chưa có job_skills.csv — chạy scrape() trước.")
else:
    skills_long = pd.read_csv("job_skills.csv")
    if skills_long.empty:
        print("job_skills.csv rỗng.")
    else:
        print(skills_long.groupby(["skill_group", "skill"]).size().sort_values(ascending=False).head(20).to_string())


skill_group    skill       
framework      React           86
language       Python          74
               Java            73
platform_tool  Git             69
language       JavaScript      66
               SQL             52
platform_tool  AWS             51
language       PHP             43
platform_tool  Docker          41
language       HTML/CSS        39
database       PostgreSQL      37
platform_tool  Android         37
language       TypeScript      35
database       MySQL           35
platform_tool  Kubernetes      34
               iOS             34
               Linux           34
               Azure           32
               Google Cloud    25
framework      Angular         24


In [14]:
# Lưu local (không phụ thuộc Google Colab)
from pathlib import Path
import shutil

out_dir = Path("output_jobs")
out_dir.mkdir(exist_ok=True)
for name in ("jobs_raw.csv", "job_skills.csv"):
    if Path(name).exists():
        shutil.copy(name, out_dir / name)
        print(f"Đã copy {name} -> {out_dir / name}")
    else:
        print(f"Thiếu {name}")


Đã copy jobs_raw.csv -> output_jobs/jobs_raw.csv
Đã copy job_skills.csv -> output_jobs/job_skills.csv


In [15]:
# Kiểm tra JOB_LINK_REGEX trên trang danh sách đầu
first = (globals().get("LIST_URLS") or [LIST_URL.format(page=1)])[0]
html = get_html(first)
if html is None:
    print("Không tải được trang danh sách.")
else:
    soup = BeautifulSoup(html, "lxml")
    rx = re.compile(JOB_LINK_REGEX)
    found = set()
    for a in soup.find_all("a", href=True):
        full = urljoin(BASE_URL, a["href"]).split("?")[0]
        if rx.search(urlparse(full).path):
            found.add(full)
    print("Số link tin tìm được trên trang 1:", len(found))
    print(*sorted(found)[:10], sep="\n")


Số link tin tìm được trên trang 1: 30
https://123job.vn/viec-lam/account-intern-yDNQM0Nyqz
https://123job.vn/viec-lam/ai-engineer-ai-technical-leader-BqP4odzJqz
https://123job.vn/viec-lam/ai-engineer-llm-engineer-JD60wdz5rn
https://123job.vn/viec-lam/android-developer-leader-ND4ZK0jJrv
https://123job.vn/viec-lam/applied-ai-engineer-ai-agent-llm-g9d1zxV8rk
https://123job.vn/viec-lam/backend-developer-golang-ArX8AeW6rV
https://123job.vn/viec-lam/chuyen-vien-c-b-v90o6x42Dk
https://123job.vn/viec-lam/data-analyst-BqP4od8Qqz
https://123job.vn/viec-lam/data-analyst-lead-A9V8Qagbqn
https://123job.vn/viec-lam/data-analyst-lead-dr8ZGP7RD4


In [16]:
# (Tùy chọn) So sánh nhanh số link dạng tin trên 123job — không bắt buộc cho scrape
html = get_html(LIST_URL.format(page=1))
if html:
    soup = BeautifulSoup(html, "lxml")
    rx = re.compile(JOB_LINK_REGEX)
    n = sum(1 for a in soup.find_all("a", href=True)
            if rx.search(urlparse(urljoin(BASE_URL, a["href"])).path))
    print("Số <a> khớp JOB_LINK_REGEX trên seed list:", n)


Số <a> khớp JOB_LINK_REGEX trên seed list: 30


In [17]:
# Không đổi tên / xóa CSV kết quả (ô cũ từng rename thành careerlink_loi_* — đã tắt)
from pathlib import Path
for f in ("jobs_raw.csv", "job_skills.csv"):
    print(f, "exists=" + str(Path(f).exists()), "size=" + (str(Path(f).stat().st_size) if Path(f).exists() else "-"))


jobs_raw.csv exists=True size=93500
job_skills.csv exists=True size=111003
